In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display
import os

plt.ioff()

## Simple way to initiate the dust corrector and get the band transmissions and wavelengths
from superbit_lensing.dust import DustCorrector

In [44]:

# --- SuperBIT filter transmissions (fractional, 0-1), wavelength already in nm ---
dust = DustCorrector()
wavelengths = dust.wavelengths.value  # nm

superbit_raw = {
    'F400W': dust._get_band_transmission('u'),
    'F480W': dust._get_band_transmission('b'),
    'F600W': dust._get_band_transmission('g'),
}
superbit_colors = {'F400W': '#8E44AD', 'F480W': '#2980B9', 'F600W': '#27AE60'}

superbit_curves = {}
for name, throughput in superbit_raw.items():
    clean = throughput > 1e-3
    superbit_curves[name] = (wavelengths[clean], throughput[clean])

# --- Roman WFI throughputs (fractional, 0-1, wavelength in nm), pre-computed from stpsf ---
# and saved to data/roman_filter/*.throughput.dat so this notebook doesn't need stpsf installed.
roman_data_dir = "/projects/mccleary_group/saha/codes/superbit-lensing/data/roman_filter"
roman_bands = ["F062", "F087", "F106", "F129", "F158", "F184", "F213"]

roman_curves = {}
for f in roman_bands:
    path = f"{roman_data_dir}/Roman_WFI.{f}.throughput.dat"
    wave_nm, throughput = np.genfromtxt(path, unpack=True)
    roman_curves[f] = (wave_nm, throughput)

roman_cmap = plt.cm.YlOrRd(np.linspace(0.35, 0.95, len(roman_bands)))
roman_colors = {f: mcolors.to_hex(c) for f, c in zip(roman_bands, roman_cmap)}

In [49]:
PROJ_DIR = os.path.dirname(os.getcwd())  # Assuming this notebook is in a subdirectory of the project
data_dir = os.path.join(PROJ_DIR, "data")

# --- Euclid NISP throughputs (T_TOTAL = full system throughput, fractional 0-1, wavelength in nm) ---
euclid_curves = {}
for b in ['Y', 'J', 'H']:
    path = f"{data_dir}/euclid_filters/NISP-PHOTO-PASSBANDS-V1-{b}_throughput.fits"
    rec = fits.getdata(path, 1)
    wave = rec['WAVE'].astype(float)
    throughput = rec['T_TOTAL']
    clean = throughput > 1e-4
    euclid_curves[b] = (wave[clean], throughput[clean])

# --- Euclid VIS throughput (fractional 0-1, wavelength in Angstrom -> nm) ---
vis_wave, vis_throughput = np.genfromtxt(f"{data_dir}/euclid_filters/Euclid_VIS.vis.dat", unpack=True)
vis_wave = vis_wave / 10.0  # Angstrom -> nm
clean = vis_throughput > 1e-3
euclid_curves['VIS'] = (vis_wave[clean], vis_throughput[clean])

euclid_colors = {'VIS': '#154360', 'Y': '#1B4F72', 'J': '#117864', 'H': '#7D6608'}

# --- LSST throughputs (fractional 0-1, wavelength in nm) ---
lsst_bands = ['u', 'g', 'r', 'i', 'z', 'y']
lsst_curves = {}
for b in lsst_bands:
    path = f"{data_dir}/lsst_filters/total_{b}.dat"
    wave, throughput = np.genfromtxt(path, comments='#', unpack=True)
    clean = throughput > 1e-3  # drop long numerical-noise tails in the raw files
    lsst_curves[b] = (wave[clean], throughput[clean])

lsst_colors = {
    'u': '#5B2C6F', 'g': '#1F618D', 'r': '#229954',
    'i': '#B7950B', 'z': '#B9770E', 'y': '#922B21',
}

# --- Unified structure: mission -> linestyle + per-band (wave, throughput, color) ---
survey_data = {
    'SuperBIT': {
        'style': '-',
        'bands': {name: dict(wave=w, throughput=t, color=superbit_colors[name])
                  for name, (w, t) in superbit_curves.items()},
    },
    'Roman': {
        'style': '--',
        'bands': {name: dict(wave=w, throughput=t, color=roman_colors[name])
                  for name, (w, t) in roman_curves.items()},
    },
    'Euclid': {
        'style': ':',
        'bands': {name: dict(wave=w, throughput=t, color=euclid_colors[name])
                  for name, (w, t) in euclid_curves.items()},
    },
    'LSST': {
        'style': '-.',
        'bands': {name: dict(wave=w, throughput=t, color=lsst_colors[name])
                  for name, (w, t) in lsst_curves.items()},
    },
}

## Choose which missions and bands to overlay

Use the selectors below to pick any combination of SuperBIT, Roman, Euclid, and LSST bands (click, or ctrl/shift-click for multiple, within each list). The plot redraws automatically, and the x- and y-axis limits are recomputed to fit exactly whatever is currently selected.

In [50]:
def nice_round(x):
    """Round to a 'nice' number so ticks read as plain, roughly equispaced integers."""
    if x < 500:
        step = 50
    elif x < 1000:
        step = 100
    elif x < 2000:
        step = 200
    else:
        step = 250
    return int(round(x / step) * step)

def nice_log_ticks(lo, hi, n=7):
    raw = np.geomspace(lo, hi, n)
    ticks = sorted(set(nice_round(x) for x in raw))
    if ticks[0] > lo:
        ticks[0] = int(round(lo / 10.0) * 10)
    return ticks

mission_toggles = {}
selectors = {}
mission_boxes = []
for mission, info in survey_data.items():
    bands = list(info['bands'].keys())
    toggle = widgets.Checkbox(value=True, description=mission, indent=False)
    sel = widgets.SelectMultiple(
        options=bands, value=tuple(bands),
        rows=min(len(bands), 6), layout=widgets.Layout(width='160px'),
    )
    mission_toggles[mission] = toggle
    selectors[mission] = sel
    mission_boxes.append(widgets.VBox([toggle, sel]))

controls = widgets.HBox(mission_boxes)
plot_output = widgets.Output()

def update_plot(*_):
    plot_output.clear_output(wait=True)

    curves = []
    for mission, sel in selectors.items():
        if not mission_toggles[mission].value:
            continue
        style = survey_data[mission]['style']
        for band in sel.value:
            info = survey_data[mission]['bands'][band]
            curves.append((info['wave'], info['throughput'], f"{mission} {band}", info['color'], style))

    with plot_output:
        if not curves:
            print("Select at least one mission/band to plot.")
            return

        lam_lo = min(wave.min() for wave, *_ in curves) * 0.9
        lam_hi = max(wave.max() for wave, *_ in curves) * 1.05
        thru_max = max(throughput.max() for _, throughput, *_ in curves)

        fig, ax = plt.subplots(figsize=(13, 6))
        for wave, throughput, label, color, ls in curves:
            ax.fill_between(wave, throughput, alpha=0.12, color=color)
            ax.plot(wave, throughput, color=color, linewidth=1.1, ls=ls, label=label)

        ax.set_xlabel(r'$\lambda$ [nm]')
        ax.set_ylabel('Transmission')
        ax.set_xlim(lam_lo, lam_hi)
        ax.set_ylim(0, min(1.05, thru_max * 1.15))
        ax.set_xscale('log')

        xticks = nice_log_ticks(lam_lo, lam_hi)
        ax.set_xticks(xticks)
        ax.set_xticklabels([str(t) for t in xticks])
        ax.minorticks_off()

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
                  ncol=min(len(curves), 7), fontsize=8.5, frameon=False)

        plt.tight_layout()
        plt.savefig("combined_filters_selected.pdf", bbox_inches="tight", dpi=600)

        # Explicit single display + close, instead of plt.show(), so the inline
        # backend's own end-of-cell auto-render can't also fire and duplicate this.
        display(fig)
        plt.close(fig)

for toggle in mission_toggles.values():
    toggle.observe(update_plot, names='value')
for sel in selectors.values():
    sel.observe(update_plot, names='value')

display(controls, plot_output)
update_plot()

Output()